In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../src')  #path to src

import pandas as pd
import os
import pycountry as pc

from data_loader import *
import data_preprocessing as dp

#  📊 Raw Data Processing | Meat Consumption Project
This notebook applies all cleaning and transformation steps to the raw datasets, based on the conclusions from the exploratory analysis (`01_data_exploration.ipynb`).

The goal is to produce a single, clean dataset ready for training.

**Summary of Processing Plan (Based on Exploratory Analysis)**


| Dataset        | Processing Actions                                                                 | Output Columns                              |
|----------------|--------------------------------------------------------------------------------------|----------------------------------------------|
| **FAOSTAT**     | - The `Value is given in ` (kg/capita/yr) <br> - Convert  `Bovine Meat`,`Mutton & Goat Meat`, `Pigmeat` & `Poultry Meat` into columns   <br> - Exclude `Item = "Meat, Other"`  <br> - Keep columns: `Area`, `Year`, `Value`  <br> - Rename: `Area → Country`, `Value → Meat_Consumption`  <br> - Aggregate by Country and Year | `Country`, `Year`, `Meat_Consumption`        |
| **GDP**         | - Drop `Unnamed: 69`  <br> - Keep only years `2010–2022` and are given as object  <br> - Rename: `Country Name → Country` - GDP appears for each year/country | `Country`, `Year`, `GDP_per_capita`          |
| **Urban**       | - Drop `Unnamed: 69`  <br> - Keep only years `2010–2022`  <br> - Rename: `Country Name → Country` | `Country`, `Year`, `Urban_Population`        |
| **Production**  |  - No processing needed  <br> - `Entity → Country`, last column contains meat production (tonnes) | `Country`, `Year`, `Meat_Production`         |
| **Education**   |- Optional dataset  <br> - Keep only `Share of population with at least some basic education`  <br> - Rename: `Entity → Country`, `Share of... → Education_Level` | `Country`, `Year`, `Education_Level`         |
| **Environment** | - Optional dataset  <br> - No processing needed  <br> - Used for environmental context only | `Product`, `Year`, `GHG_Emission_per_kg`     |



## Data processing

In this section, we clean and harmonize the raw datasets and merge them into a single DataFrame suitable for training. This includes:

- Filtering and renaming key columns
- Selecting overlapping years across datasets
- Aligning countries and handling inconsistencies
- Merging all features into a single dataframe

The processed data is exported as `meat_processed_merged_data.csv`.


In [3]:
dfs = load_raw_data()

### Process FAOSTAT Dataset

In [4]:
faostat = dfs['faostat']

**FAOSTAT – Processing steps**

- Filtered by `Element = "Food supply quantity (kg/capita/yr)"`.
- Removed `Item = "Meat, Other"`.
- Grouped by `Country` and `Year` to compute total meat consumption per capita.


In [5]:
# Filtering by 'Element=="Food supply quantity (kg/capita/yr)"'
faostat_filtered = faostat[faostat['Element'] == "Food supply quantity (kg/capita/yr)"]
# We drop all the dataset corresponding to 'Meat,Other'
faostat_filtered = faostat_filtered[faostat_filtered['Item'] != 'Meat, Other']
# Renamin columns
faostat_filtered = faostat_filtered.rename(columns={'Area':'Country','Value':'meat_consumption'})

In [6]:
define_columns = ['Bovine_Meat','Poultry_Meat','Mutton_Goat_Meat', 'PigMeat']
faostat_filtered = faostat_filtered.pivot_table(
    index   = ['Country', 'Year'],
    columns = 'Item',  #to convert in new column,
    values  = 'meat_consumption', # values in the column
    aggfunc = 'sum' # In case of duplicates
    
).reset_index()

### Process PDG Dataset

In [7]:
gdp = dfs['gdp']

In [8]:
# Renaming column
gdp = gdp.rename(columns={'Country Name':'Country'})

In [9]:
#Defining the columns we need to use from this file
columns = ['Country']+[str(year) for year in range(2010,2023)]

In [10]:
# Melting the required data from 2010 to 2022
gdp_filtered = gdp[columns ].melt(id_vars='Country', var_name='Year', value_name='GDP_per_capita')


In [11]:
gdp_filtered

,Country,Year,GDP_per_capita
0,Aruba,2010,24093.140151
1,Africa Eastern and Southern,2010,1601.727651
2,Afghanistan,2010,560.621505
3,Africa Western and Central,2010,1663.966937
4,Angola,2010,3597.342932
...,...,...,...
3453,Kosovo,2022,5290.947472
3454,"Yemen, Rep.",2022,615.702078
3455,South Africa,2022,6523.410978
3456,Zambia,2022,1447.123101


Years are introduced as `object`

In [12]:
gdp_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country         3458 non-null   object 
 1   Year            3458 non-null   object 
 2   GDP_per_capita  3369 non-null   float64
dtypes: float64(1), object(2)
memory usage: 81.2+ KB


### Process Urban Dataset

In [13]:
urban = dfs['urban']

In [14]:
# Renaming column
urban = urban.rename(columns={'Country Name':'Country'})

In [15]:
#Defining the columns we need to use from this file
columns = ['Country']+[str(year) for year in range(2010,2023)]

In [16]:
# Melting the required data from 2010 to 2022
urban_filtered = urban[columns ].melt(id_vars='Country', var_name='Year', value_name='urban_per_capita')

In [17]:
# Checking output
urban_filtered

,Country,Year,urban_per_capita
0,Aruba,2010,43.059000
1,Africa Eastern and Southern,2010,32.195595
2,Afghanistan,2010,23.737000
3,Africa Western and Central,2010,41.672423
4,Angola,2010,59.783000
...,...,...,...
3453,Kosovo,2022,NaN
3454,"Yemen, Rep.",2022,39.188000
3455,South Africa,2022,68.335000
3456,Zambia,2022,45.761000


Years are introduced as `object`

In [18]:
urban_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Country           3458 non-null   object 
 1   Year              3458 non-null   object 
 2   urban_per_capita  3419 non-null   float64
dtypes: float64(1), object(2)
memory usage: 81.2+ KB


### Process Production Dataset

In [56]:
production= dfs['production']

In [57]:
#Checking columns for rename
production.columns

Index(['Entity', 'Code', 'Year',
       'Meat, total | 00001765 || Production | 005510 || tonnes'],
      dtype='object')

In [58]:
renamedu = {'Entity':'Country',  'Meat, total | 00001765 || Production | 005510 || tonnes': 'production_tones'}

In [59]:
production = production.rename(columns=renamedu)


In [62]:
# Dropping column `Code`. I could use drop(`Code`), but since we have a few columns we can aso call what we need.
production_filtered = production[['Country', 'Year','production_tones']]
production_filtered

,Country,Year,production_tones
0,Afghanistan,1961,129420.00
1,Afghanistan,1962,132205.73
2,Afghanistan,1963,138971.36
3,Afghanistan,1964,143830.00
4,Afghanistan,1965,150195.00
...,...,...,...
14609,Zimbabwe,2019,823826.10
14610,Zimbabwe,2020,821114.40
14611,Zimbabwe,2021,895893.50
14612,Zimbabwe,2022,926426.20


In [63]:
production_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14614 entries, 0 to 14613
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Country           14614 non-null  object 
 1   Year              14614 non-null  int64  
 2   production_tones  14614 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 342.6+ KB


### Merging datasets
**Dataset Preparation for Merging**

Before merging:
- we ensure the `Year` column has consistent data types across all datasets.  
Some datasets may have `Year` as `object` after melting (e.g., GDP, Urban), so we convert it to `int`.
- we ensure that countries are given with the same names


In [28]:
# Convertinv (objects) into (strings)
gdp_filtered['Year'] = gdp_filtered['Year'].astype(int)
urban_filtered['Year'] = urban_filtered['Year'].astype(int)

In [ ]:
# Applying  standard names to my dataset
faostat_filtered = dp.standardize_country_names(faostat_filtered)
gdp_filtered = dp.standardize_country_names(gdp_filtered)
urban_filtered = dp.standardize_country_names(urban_filtered)
production_filtered= dp.standardize_country_names(production_filtered)

Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not found in regex
Africa not f

In [65]:
dp.comparing_name_country(faostat_filtered,production_filtered, name1 = 'faostat_filtered',name2='production_filtered')



faostat_filtered countries not in production_filtered (1):
{'Marshall Islands'}


{'Marshall Islands'}

In [66]:
merge1 = dp.mergingfunc(faostat_filtered,gdp_filtered)
merge2 = dp.mergingfunc(merge1, urban_filtered)
merge_df = dp.mergingfunc(merge2,production_filtered)

In [67]:
merge_df

,Country,Year,Bovine Meat,Mutton & Goat Meat,Pigmeat,Poultry Meat,GDP_per_capita,urban_per_capita,production_tones
0,Afghanistan,2010,4.74,5.04,NaN,2.29,560.621505,23.737000,328160.0
1,Afghanistan,2010,4.74,5.04,NaN,2.29,560.621505,51.570686,328160.0
2,Afghanistan,2010,4.74,5.04,NaN,2.29,560.621505,48.784801,328160.0
3,Afghanistan,2010,4.74,5.04,NaN,2.29,560.621505,48.617382,328160.0
4,Afghanistan,2010,4.74,5.04,NaN,2.29,4858.243167,23.737000,328160.0
...,...,...,...,...,...,...,...,...,...
2604,Zimbabwe,2018,40.99,1.82,0.52,4.38,2271.852504,32.209000,771502.0
2605,Zimbabwe,2019,40.98,1.77,0.69,4.47,1683.913136,32.210000,823826.1
2606,Zimbabwe,2020,40.12,1.76,0.64,7.21,1730.453910,32.242000,821114.4
2607,Zimbabwe,2021,43.77,1.95,0.73,7.33,1724.387271,32.303000,895893.5
